# التحدي: تحليل نص حول علم البيانات

في هذا المثال، دعنا نقوم بتمرين بسيط يغطي جميع خطوات عملية علم البيانات التقليدية. لا يتعين عليك كتابة أي كود، يمكنك فقط النقر على الخلايا أدناه لتشغيلها ومراقبة النتيجة. كتحدي، يُشجعك على تجربة هذا الكود مع بيانات مختلفة.

## الهدف

في هذا الدرس، ناقشنا مفاهيم مختلفة تتعلق بعلم البيانات. دعنا نحاول اكتشاف المزيد من المفاهيم ذات الصلة من خلال القيام ببعض **التنقيب في النصوص**. سنبدأ بنص عن علم البيانات، نستخرج منه الكلمات المفتاحية، ثم نحاول تصور النتيجة.

كنص، سأستخدم الصفحة حول علم البيانات من ويكيبيديا:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## الخطوة 1: الحصول على البيانات

الخطوة الأولى في كل عملية علوم بيانات هي الحصول على البيانات. سنستخدم مكتبة `requests` للقيام بذلك:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## الخطوة 2: تحويل البيانات

الخطوة التالية هي تحويل البيانات إلى الشكل المناسب للمعالجة. في حالتنا، قمنا بتنزيل شفرة HTML المصدر من الصفحة، ونحتاج إلى تحويلها إلى نص عادي.

هناك العديد من الطرق التي يمكن أن يتم بها ذلك. سنستخدم [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/)، مكتبة بايثون شهيرة لتحليل HTML. يتيح لنا BeautifulSoup استهداف عناصر HTML محددة، بحيث يمكننا التركيز على محتوى المقال الرئيسي من ويكيبيديا وتقليل بعض قوائم الملاحة، والأشرطة الجانبية، والتذييلات، ومحتويات أخرى غير ذات صلة (على الرغم من أنه قد يبقى بعض النصوص الثابتة).


أولاً، نحتاج إلى تثبيت مكتبة BeautifulSoup لتحليل HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## الخطوة 3: الحصول على رؤى

الخطوة الأهم هي تحويل بياناتنا إلى شكل يمكننا من خلاله استخلاص الرؤى. في حالتنا، نريد استخراج الكلمات المفتاحية من النص، ومعرفة أي الكلمات المفتاحية هي الأكثر معنى.

سنستخدم مكتبة بايثون تسمى [RAKE](https://github.com/aneesha/RAKE) لاستخراج الكلمات المفتاحية. أولاً، دعونا نثبت هذه المكتبة في حال لم تكن موجودة:


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

الوظيفة الرئيسية متاحة من كائن `Rake`، والذي يمكننا تخصيصه باستخدام بعض المعايير. في حالتنا، سنحدد الحد الأدنى لطول الكلمة المفتاحية ليكون 5 أحرف، والحد الأدنى لتكرار الكلمة المفتاحية في المستند ليكون 3، والحد الأقصى لعدد الكلمات في الكلمة المفتاحية ليكون 2. لا تتردد في تجربة قيم أخرى وملاحظة النتيجة.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


حصلنا على قائمة بالمصطلحات مع درجة الأهمية المرتبطة بها. كما ترون، فإن التخصصات الأكثر صلة، مثل تعلم الآلة والبيانات الضخمة، موجودة في القائمة في المراتب العليا.

## الخطوة 4: تصور النتيجة

يمكن للناس تفسير البيانات بشكل أفضل في الشكل المرئي. لذلك غالبًا ما يكون من المنطقي تصور البيانات من أجل استخلاص بعض الرؤى. يمكننا استخدام مكتبة `matplotlib` في بايثون لرسم توزيع بسيط للكلمات المفتاحية مع صلتها:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

هناك، مع ذلك، طريقة أفضل لتصور تردد الكلمات - باستخدام **سحابة الكلمات**. سنحتاج إلى تثبيت مكتبة أخرى لرسم سحابة الكلمات من قائمة الكلمات الرئيسية لدينا.


In [ ]:
!{sys.executable} -m pip install wordcloud

كائن `WordCloud` مسؤول عن استلام إما النص الأصلي، أو قائمة بالكلمات محسوبة مسبقًا مع تكراراتها، ويُرجع صورة، والتي يمكن عرضها باستخدام `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

يمكننا أيضًا تمرير النص الأصلي إلى `WordCloud` - دعونا نرى إذا كنا قادرين على الحصول على نتيجة مماثلة:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

يمكنك أن ترى أن سحابة الكلمات تبدو الآن أكثر إثارة للإعجاب، لكنها تحتوي أيضًا على الكثير من الضوضاء (مثل الكلمات غير ذات الصلة مثل `Retrieved on`). كما أننا نحصل على عدد أقل من الكلمات المفتاحية التي تتكون من كلمتين، مثل *عالم البيانات*، أو *علوم الحاسوب*. ويرجع ذلك إلى أن خوارزمية RAKE تقوم بعمل أفضل بكثير في اختيار الكلمات المفتاحية الجيدة من النص. يوضح هذا المثال أهمية معالجة البيانات وتنقيتها مسبقًا، لأن الصورة الواضحة في النهاية ستسمح لنا باتخاذ قرارات أفضل.

في هذا التمرين، مررنا بعملية بسيطة لاستخلاص بعض المعاني من نص ويكيبيديا، في شكل كلمات مفتاحية وسحابة كلمات. هذا المثال بسيط جدًا، لكنه يُظهِر جيدًا كل الخطوات النموذجية التي يتخذها عالم البيانات عند العمل مع البيانات، بدءًا من الحصول على البيانات، ووصولًا إلى التمثيل البياني.

في دورتنا، سنناقش كل تلك الخطوات بالتفصيل.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**تنويه**:
تمت ترجمة هذا المستند باستخدام خدمة الترجمة بالذكاء الاصطناعي [Co-op Translator](https://github.com/Azure/co-op-translator). بينما نسعى للدقة، يرجى العلم أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الرسمي والمعتمد. للمعلومات الهامة، يُنصح بالاستعانة بترجمة بشرية محترفة. نحن غير مسؤولين عن أي سوء فهم أو تفسير ناتج عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
